# nnU-Net Baseline

---

##Objective
This notebook represents **Phase 1** of the thesis project. The goal is to train a state-of-the-art cardiac segmentation network on the ACDC dataset and obtain baseline results — i.e., results with the original network, without any modifications for homomorphic encryption.These results will serve as the reference point to measure the degradation introduced in subsequent phases:

**Phase 2**: HE-friendly version of the network (replacing operations incompatible with HE)
**Phase 3**: encrypted inference with OpenFHE/CKKS

---

## Dataset: ACDC
The ACDC dataset (Automated Cardiac Diagnosis Challenge) is the world reference benchmark for cardiac MRI segmentation. It contains images from 150 patients acquired at the University Hospital of Dijon with two Siemens scanners (1.5T and 3.0T).
In this notebook we use the **100 patients from the official training set**. For each patient, two frames are used:

- **ED** (End-Diastole): heart at maximum volume, larger and easier structures to segment
- **ES **(End-Systole): heart at minimum volume, smaller structures

This results in a training set of **200 cases total**.
The three structures to segment, in order of increasing difficulty:

- **LV** (Left Ventricle): regular circular shape, easiest
- **MYO** (Myocardium): thin ring around the LV, medium difficulty
- **RV** (Right Ventricle): variable crescent shape, hardest

---

## Preprocessing

Preprocessing is entirely delegated to nnU-Net, which operates in four automatic phases:

**Fingerprinting**: dataset analysis to extract average dimensions, spacing, and class distribution. On ACDC the in-plane resolution is ~1.5×1.5 mm with slice thickness ~9 mm (anisotropic images), which justifies 2D slice-by-slice segmentation.

**Planning**: nnU-Net autonomously decides the optimal architecture, patch size, and batch size based on available VRAM. On this dataset with a T4 GPU it chose patch 256×224 and batch size 56.

**Preprocessing**: resampling of all images to the same spacing (1.5625×1.5625 mm), Z-score normalization computed on non-background voxels, saving in optimized format.

The train/validation split was defined **per patient**: if a patient goes into validation, all their frames (ED and ES) go into validation together. This guarantees no data leakage between training and validation. The split uses fixed seed 1234 for reproducibility: 160 cases in train, 40 in validation for fold 0.

---

## Network Architecture

nnU-Net automatically configures a **2D U-Net** with the following structure on ACDC:

```
Input: patch 256×224, 1 channel (cine MRI)

ENCODER (6 stage):
  Stage 0: Conv3×3 → InstanceNorm → LeakyReLU × 2   (32 filtri,  stride 1×1)
  Stage 1: Conv3×3 → InstanceNorm → LeakyReLU × 2   (64 filtri,  stride 2×2)
  Stage 2: Conv3×3 → InstanceNorm → LeakyReLU × 2   (128 filtri, stride 2×2)
  Stage 3: Conv3×3 → InstanceNorm → LeakyReLU × 2   (256 filtri, stride 2×2)
  Stage 4: Conv3×3 → InstanceNorm → LeakyReLU × 2   (512 filtri, stride 2×2)
  Stage 5: Conv3×3 → InstanceNorm → LeakyReLU × 2   (512 filtri, stride 2×2)

Skip connections: link each encoder stage to the corresponding decoder stage

DECODER (5 stage):
  Upsample → Skip concatenation → Conv3×3 × 2 (with  InstanceNorm e LeakyReLU)

Output: segmentation map 256×224, 4 classes (background=0, RV=1, MYO=2, LV=3)
```

Downsampling is performed via **stride convolution** (stride 2×2) instead of MaxPool — this is relevant for the HE phase because convolutions are linear operations, while MaxPool requires value comparison (non-linear and incompatible with HE).

The loss function is the **Dice + Cross-Entropy** combination, standard in nnU-Net, computed in plaintext during training.

---

## Homomorphic Encryption Compatibility

Analyzing the architecture from the perspective of homomorphic encryption (CKKS scheme), operations divide into:

| Component | HE Compatible | Reason |
|---|---|---|
| Conv2d 3×3 | ✅ Yes | Linear operation |
| Skip connections | ✅ Yes | Simple concatenation |
| Stride convolution | ✅ Yes | Linear operation |
| LeakyReLU | ❌ No | Non-linear → replace with x² |
| InstanceNorm | ❌ No | Requires square root → remove |
| Softmax output | ❌ No | Not needed in HE: decrypt first, then apply in plaintext |


---

## Evaluation Metrics

Results are measured with two standard metrics on ACDC:

**Dice Similarity Coefficient (DSC)**: measures the overlap between the predicted mask and the ground truth. Ranges from 0 (no overlap) to 1 (perfect overlap). It is the primary metric — all reference papers (nnU-Net, DAFNet, ASM-UNet) report it.

Hausdorff Distance at 95th percentile (HD95): measures the maximum distance (at the 95th percentile, to exclude outliers) between the borders of the predicted mask and the ground truth, in millimeters. Complementary to Dice because it captures severe boundary errors that Dice can mask.
Reference values from the literature (nnU-Net, fold 0 on ACDC):

- LV: Dice ~0.940 (RV: ~0.946)
- MYO: Dice ~0.901 (MYO: ~0.896)
- RV: Dice ~0.912 (LV: ~0.967)

---



In [1]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no GPU detected. Go to Runtime -> Change runtime type -> GPU')

PyTorch version: 2.10.0+cu128
CUDA disponibile: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
!pip install nnunetv2 -q
!pip install nibabel -q
print('Installation done.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.7/100.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive done.')

Mounted at /content/drive
Google Drive montato.


In [4]:
import os

ACDC_RAW = '/content/drive/MyDrive/tesi/tesi_acdc/training'

if os.path.exists(ACDC_RAW):
    patients = sorted([p for p in os.listdir(ACDC_RAW) if p.startswith('patient')])
    print(f'Found {len(patients)} patients')
    print(f'First: {patients[0]}, Last: {patients[-1]}')
else:
    print(f'ERROR: path not found: {ACDC_RAW}')
    print('Make sure tesi_acdc is inside MyDrive/tesi/')

Trovati 100 pazienti
Primo: patient001, Ultimo: patient100


In [5]:
# Show all files for patient001
# Expected: Info.cfg, frame01.nii.gz, frame01_gt.nii.gz, frame12.nii.gz, frame12_gt.nii.gz, 4d.nii.gz
patient_example = os.path.join(ACDC_RAW, 'patient001')
print('Files in patient001:')
for f in sorted(os.listdir(patient_example)):
    print(f'  {f}')

File in patient001:
  .DS_Store
  Info.cfg
  MANDATORY_CITATION.md
  patient001_4d.nii.gz
  patient001_frame01.nii.gz
  patient001_frame01_gt.nii.gz
  patient001_frame12.nii.gz
  patient001_frame12_gt.nii.gz


In [6]:
import os

BASE_DRIVE = '/content/drive/MyDrive/tesi/nnunet_workspace'

nnUNet_raw          = os.path.join(BASE_DRIVE, 'nnUNet_raw')
nnUNet_preprocessed = os.path.join(BASE_DRIVE, 'nnUNet_preprocessed')
nnUNet_results      = os.path.join(BASE_DRIVE, 'nnUNet_results')

for d in [nnUNet_raw, nnUNet_preprocessed, nnUNet_results]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = nnUNet_raw
os.environ['nnUNet_preprocessed'] = nnUNet_preprocessed
os.environ['nnUNet_results']      = nnUNet_results

print('Directory nnU-Net:')
print(f'  raw:          {nnUNet_raw}')
print(f'  preprocessed: {nnUNet_preprocessed}')
print(f'  results:      {nnUNet_results}')

Directory nnU-Net:
  raw:          /content/drive/MyDrive/tesi/nnunet_workspace/nnUNet_raw
  preprocessed: /content/drive/MyDrive/tesi/nnunet_workspace/nnUNet_preprocessed
  results:      /content/drive/MyDrive/tesi/nnunet_workspace/nnUNet_results


## ACDC → nnU-Net format conversion

We follow exactly the official nnU-Net script `Dataset027_ACDC.py`.

**`copy_files`**: copies all frames per patient (ED, ES, others)
- Excludes only `_4d` files (full 4D volume, too large)
- Excludes `_gt` files from images, places them in labelsTr
- Naming: `patient001_frame01_0000.nii.gz` (same as the official script)

**`create_ACDC_split`**: creates the train/val split per patient
- Groups by patient, not by frame
- If patient001 goes to val, both their ED and ES go to val
- This avoids data leakage between train and validation
- Fixed seed (1234) for reproducibility

In [7]:
import os
import shutil
import json
import numpy as np
from pathlib import Path


def copy_files(src_training_folder, train_dir, labels_dir):
    """
    Adattato da Dataset027_ACDC.py ufficiale di nnU-Net.

    Copia tutti i frame per paziente (ED, ES, eventualmente altri).
    Esclude: file _4d (volume 4D completo) e file non .nii.gz
    Le immagini vanno in imagesTr con suffisso _0000.
    Le maschere (_gt) vanno in labelsTr senza il suffisso _gt.
    """
    patients = sorted([f for f in Path(src_training_folder).iterdir() if f.is_dir()])
    num_cases = 0

    for patient_dir in patients:
        for file in sorted(patient_dir.iterdir()):
            if file.suffix != '.gz':
                continue
            if '_4d' in file.name:
                # Salta il volume 4D — non serve per la segmentazione 2D slice
                continue

            if '_gt' not in file.name:
                # Immagine: patient001_frame01.nii.gz -> patient001_frame01_0000.nii.gz
                # file.stem = 'patient001_frame01.nii', split('.')[0] = 'patient001_frame01'
                case_name = file.stem.split('.')[0]
                dst = os.path.join(train_dir, f'{case_name}_0000.nii.gz')
                shutil.copy2(file, dst)
                num_cases += 1
            else:
                # Maschera: patient001_frame01_gt.nii.gz -> patient001_frame01.nii.gz
                dst_name = file.name.replace('_gt', '')
                shutil.copy2(file, os.path.join(labels_dir, dst_name))

    return num_cases


def create_ACDC_split(labels_dir, seed=1234):
    """
    Identico a create_ACDC_split in Dataset027_ACDC.py ufficiale.

    Raggruppa i casi per paziente (patient001, patient002, ...)
    e crea 5 fold dove ogni fold ha pazienti interi in val.
    Questo garantisce che ED e ES dello stesso paziente
    siano sempre nello stesso split (no data leakage).
    """
    # Lista tutti i file .nii.gz in labelsTr
    nii_files = sorted([f for f in os.listdir(labels_dir) if f.endswith('.nii.gz')])

    # Estrai i nomi paziente unici (es. 'patient001' da 'patient001_frame01.nii.gz')
    patients = np.unique([f[:len('patient001')] for f in nii_files])

    rs = np.random.RandomState(seed)
    rs.shuffle(patients)

    splits = []
    for fold in range(5):
        val_patients   = patients[fold::5]
        train_patients = [p for p in patients if p not in val_patients]

        # Un caso e' il nome file senza .nii.gz
        val_cases   = [f[:-7] for f in nii_files
                       for vp in val_patients if f.startswith(vp)]
        train_cases = [f[:-7] for f in nii_files
                       for tp in train_patients if f.startswith(tp)]

        splits.append({'train': train_cases, 'val': val_cases})

    return splits


# --- Esecuzione ---

dataset_folder = os.path.join(nnUNet_raw, 'Dataset027_ACDC')
images_tr = os.path.join(dataset_folder, 'imagesTr')
labels_tr = os.path.join(dataset_folder, 'labelsTr')

for d in [images_tr, labels_tr]:
    os.makedirs(d, exist_ok=True)

print('Copia file in corso...')
num_cases = copy_files(ACDC_RAW, images_tr, labels_tr)

# dataset.json
dataset_json = {
    'channel_names': {'0': 'cineMRI'},
    'labels': {
        'background': 0,
        'RV':  1,
        'MYO': 2,
        'LV':  3
    },
    'numTraining': num_cases,
    'file_ending': '.nii.gz',
    'name': 'ACDC',
    'description': 'Automated Cardiac Diagnosis Challenge'
}
with open(os.path.join(dataset_folder, 'dataset.json'), 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f'Casi copiati: {num_cases}')
print(f'Dataset in: {dataset_folder}')

Copia file in corso...
Casi copiati: 200
Dataset in: /content/drive/MyDrive/tesi/nnunet_workspace/nnUNet_raw/Dataset027_ACDC


In [8]:
# Crea lo split train/val per paziente e salvalo
# Questo file viene letto da nnU-Net durante il training
# e garantisce che ED e ES dello stesso paziente siano sempre nello stesso fold

preprocessed_dataset = os.path.join(nnUNet_preprocessed, 'Dataset027_ACDC')
os.makedirs(preprocessed_dataset, exist_ok=True)

splits = create_ACDC_split(labels_tr, seed=1234)

splits_path = os.path.join(preprocessed_dataset, 'splits_final.json')
with open(splits_path, 'w') as f:
    json.dump(splits, f, indent=2)

print(f'Split salvato in: {splits_path}')
print(f'\nFold 0:')
print(f'  Train: {len(splits[0]["train"])} casi')
print(f'  Val:   {len(splits[0]["val"])} casi')
print(f'\nEsempio val fold 0 (primi 4): {splits[0]["val"][:4]}')
print('-> ED e ES dello stesso paziente sono sempre insieme')

Split salvato in: /content/drive/MyDrive/tesi/nnunet_workspace/nnUNet_preprocessed/Dataset027_ACDC/splits_final.json

Fold 0:
  Train: 160 casi
  Val:   40 casi

Esempio val fold 0 (primi 4): ['patient005_frame01', 'patient005_frame13', 'patient007_frame01', 'patient007_frame07']
-> ED e ES dello stesso paziente sono sempre insieme


In [9]:
n_tr_img = len(os.listdir(images_tr))
n_tr_lbl = len(os.listdir(labels_tr))

print(f'imagesTr: {n_tr_img} file')
print(f'labelsTr: {n_tr_lbl} file')
print(f'\nPrimi 4 file imagesTr:')
for f in sorted(os.listdir(images_tr))[:4]:
    print(f'  {f}')
print(f'\nPrimi 4 file labelsTr:')
for f in sorted(os.listdir(labels_tr))[:4]:
    print(f'  {f}')

imagesTr: 200 file
labelsTr: 200 file

Primi 4 file imagesTr:
  patient001_frame01_0000.nii.gz
  patient001_frame12_0000.nii.gz
  patient002_frame01_0000.nii.gz
  patient002_frame12_0000.nii.gz

Primi 4 file labelsTr:
  patient001_frame01.nii.gz
  patient001_frame12.nii.gz
  patient002_frame01.nii.gz
  patient002_frame12.nii.gz


## Plan and Preprocess

nnU-Net automatically analyzes the dataset and decides:
- Patch size and batch size based on available VRAM
- Architecture (number of layers, filters)
- Resampling spacing
- Normalization strategy


In [10]:
!nnUNetv2_train 27 2d 0 --npz -num_epochs 50

usage: nnUNetv2_train [-h] [-tr TR] [-p P]
                      [-pretrained_weights PRETRAINED_WEIGHTS]
                      [-num_gpus NUM_GPUS] [--npz] [--c] [--val] [--val_best]
                      [--disable_checkpointing] [-device DEVICE]
                      dataset_name_or_id configuration fold
nnUNetv2_train: error: unrecognized arguments: -num_epochs 50


In [11]:
import json, os

plans_path = os.path.join(nnUNet_preprocessed, 'Dataset027_ACDC', 'nnUNetPlans.json')
with open(plans_path, 'r') as f:
    plans = json.load(f)

cfg = plans['configurations']['2d']

print('=== Piano nnU-Net 2D ===')
print(f"Patch size:  {cfg['patch_size']}")
print(f"Batch size:  {cfg['batch_size']}")
print()
print('Tutte le chiavi disponibili:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

=== Piano nnU-Net 2D ===
Patch size:  [256, 224]
Batch size:  56

Tutte le chiavi disponibili:
  data_identifier: nnUNetPlans_2d
  preprocessor_name: DefaultPreprocessor
  batch_size: 56
  patch_size: [256, 224]
  median_image_size_in_voxels: [237.0, 208.0]
  spacing: [1.5625, 1.5625]
  normalization_schemes: ['ZScoreNormalization']
  use_mask_for_norm: [False]
  resampling_fn_data: resample_data_or_seg_to_shape
  resampling_fn_seg: resample_data_or_seg_to_shape
  resampling_fn_data_kwargs: {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}
  resampling_fn_seg_kwargs: {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}
  resampling_fn_probabilities: resample_data_or_seg_to_shape
  resampling_fn_probabilities_kwargs: {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}
  architecture: {'network_class_name': 'dynamic_network_architectures.architectures.unet.PlainConvUNet', 'arch_kwargs': {'n_stages': 6, 'features_per_stage': [32, 64, 1

In [12]:
# Analisi class imbalance del dataset ACDC

import nibabel as nib

labels_dir = os.path.join(nnUNet_raw, 'Dataset027_ACDC', 'labelsTr')
counts = {0: 0, 1: 0, 2: 0, 3: 0}
names  = {0: 'Background', 1: 'RV', 2: 'MYO', 3: 'LV'}

files = sorted(os.listdir(labels_dir))
for fname in files:
    data = nib.load(os.path.join(labels_dir, fname)).get_fdata().flatten()
    for label in [0, 1, 2, 3]:
        counts[label] += int(np.sum(data == label))

total = sum(counts.values())
print('=== Distribuzione classi ===')
for label, count in counts.items():
    print(f'  {names[label]:12s}: {count:15,}  ({count/total*100:.2f}%)')
print(f'\n  Totale voxel: {total:,}')

=== Distribuzione classi ===
  Background  :      98,189,937  (96.21%)
  RV          :       1,236,051  (1.21%)
  MYO         :       1,328,344  (1.30%)
  LV          :       1,298,596  (1.27%)

  Totale voxel: 102,052,928


## Baseline Training

- Configuration: **2D**
- Fold: **0**
- Epochs: 1000 (nnU-Net default) (currently using 150)
- Checkpoints are saved to Drive every 50 epochs

In [13]:
# Copia dati preprocessati in RAM locale (molto più veloce di Drive)
print("Copia in corso... (5-10 minuti)")
!cp -r /content/drive/MyDrive/tesi/nnunet_workspace/nnUNet_preprocessed /content/nnUNet_preprocessed
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
print("Copia completata.")

Copia in corso... (5-10 minuti)
Copia completata.


In [14]:
import os
import pickle
import numpy as np

preprocessed_dir = os.path.join(nnUNet_preprocessed, 'Dataset027_ACDC', 'nnUNetPlans_2d')

# Leggi le dimensioni dai file .pkl che contengono i metadati
pkl_files = sorted([f for f in os.listdir(preprocessed_dir) if f.endswith('.pkl') and 'seg' not in f])

shapes = []
for f in pkl_files[:10]:  # primi 10 per velocità
    with open(os.path.join(preprocessed_dir, f), 'rb') as fp:
        metadata = pickle.load(fp)
    print(f'{f}: {metadata}')

patient001_frame01.pkl: {'sitk_stuff': {'spacing': (1.5625, 1.5625, 10.0), 'origin': (0.0, 0.0, 0.0), 'direction': (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)}, 'spacing': [np.float64(10.0), np.float64(1.5625), np.float64(1.5625)], 'shape_before_cropping': (10, 256, 216), 'bbox_used_for_cropping': [[0, 10], [0, 256], [0, 216]], 'shape_after_cropping_and_before_resampling': (10, 256, 216), 'class_locations': {1: array([[  0,   1, 157,  63],
       [  0,   2, 128,  63],
       [  0,   1, 157,  88],
       ...,
       [  0,   1, 163,  71],
       [  0,   1, 145,  71],
       [  0,   3, 128,  64]]), 2: array([[  0,   2, 142, 124],
       [  0,   1, 118,  87],
       [  0,   5, 138, 131],
       ...,
       [  0,   6, 105, 109],
       [  0,   4, 103, 110],
       [  0,   1, 111, 117]]), 3: array([[  0,   1, 116,  91],
       [  0,   1, 128,  93],
       [  0,   6, 145, 101],
       ...,
       [  0,   6, 121,  83],
       [  0,   2, 136,  87],
       [  0,   8, 135, 103]])}}
patient001_f

In [15]:
import os

preprocessed_dir = os.path.join(nnUNet_preprocessed, 'Dataset027_ACDC', 'nnUNetPlans_2d')

print(f'Path: {preprocessed_dir}')
print(f'Esiste: {os.path.exists(preprocessed_dir)}')

if os.path.exists(preprocessed_dir):
    files = os.listdir(preprocessed_dir)
    print(f'Totale file: {len(files)}')
    print(f'Primi 5: {files[:5]}')
    npy = [f for f in files if f.endswith('.npy')]
    pkl = [f for f in files if f.endswith('.pkl')]
    print(f'File .npy: {len(npy)}')
    print(f'File .pkl: {len(pkl)}')

Path: /content/drive/MyDrive/tesi/nnunet_workspace/nnUNet_preprocessed/Dataset027_ACDC/nnUNetPlans_2d
Esiste: True
Totale file: 600
Primi 5: ['patient001_frame01.b2nd', 'patient001_frame12.b2nd', 'patient002_frame12.b2nd', 'patient002_frame01.b2nd', 'patient001_frame01_seg.b2nd']
File .npy: 0
File .pkl: 200


In [16]:
import os, pickle
import numpy as np

preprocessed_dir = os.path.join(nnUNet_preprocessed, 'Dataset027_ACDC', 'nnUNetPlans_2d')

pkl_files = sorted([f for f in os.listdir(preprocessed_dir)
                    if f.endswith('.pkl') and 'seg' not in f])

shapes, spacings = [], []
for pkl_f in pkl_files[:20]:
    with open(os.path.join(preprocessed_dir, pkl_f), 'rb') as fp:
        meta = pickle.load(fp)
    shapes.append(meta['shape_after_cropping_and_before_resampling'])
    spacings.append(meta['spacing'])

print(f'Shape tipica dopo crop (prima di resample): {shapes[0]}')
print(f'Spacing dopo resampling: {spacings[0]}')
print(f'Range H: {min(s[1] for s in shapes)} – {max(s[1] for s in shapes)}')
print(f'Range W: {min(s[2] for s in shapes)} – {max(s[2] for s in shapes)}')
print(f'\nEsempio metadati completi primo caso:')
with open(os.path.join(preprocessed_dir, pkl_files[0]), 'rb') as fp:
    meta = pickle.load(fp)
print(meta)

Shape tipica dopo crop (prima di resample): (10, 256, 216)
Spacing dopo resampling: [np.float64(10.0), np.float64(1.5625), np.float64(1.5625)]
Range H: 216 – 256
Range W: 199 – 256

Esempio metadati completi primo caso:
{'sitk_stuff': {'spacing': (1.5625, 1.5625, 10.0), 'origin': (0.0, 0.0, 0.0), 'direction': (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)}, 'spacing': [np.float64(10.0), np.float64(1.5625), np.float64(1.5625)], 'shape_before_cropping': (10, 256, 216), 'bbox_used_for_cropping': [[0, 10], [0, 256], [0, 216]], 'shape_after_cropping_and_before_resampling': (10, 256, 216), 'class_locations': {1: array([[  0,   1, 157,  63],
       [  0,   2, 128,  63],
       [  0,   1, 157,  88],
       ...,
       [  0,   1, 163,  71],
       [  0,   1, 145,  71],
       [  0,   3, 128,  64]]), 2: array([[  0,   2, 142, 124],
       [  0,   1, 118,  87],
       [  0,   5, 138, 131],
       ...,
       [  0,   6, 105, 109],
       [  0,   4, 103, 110],
       [  0,   1, 111, 117]]), 3: array

In [17]:
# Verifica i tempi reali
import time, pickle, os
import numpy as np

preprocessed_dir = os.path.join(nnUNet_preprocessed, 'Dataset027_ACDC', 'nnUNetPlans_2d')
pkl_files = sorted([f for f in os.listdir(preprocessed_dir) if f.endswith('.pkl') and 'seg' not in f])

start = time.time()
for f in pkl_files[:10]:
    with open(os.path.join(preprocessed_dir, f), 'rb') as fp:
        pickle.load(fp)
print(f'Tempo medio lettura pkl: {(time.time()-start)/10*1000:.0f}ms')

Tempo medio lettura pkl: 4ms


In [25]:
# Bounding box e costo del crop — implementazione diretta senza import nnunetv2
import time, nibabel as nib, numpy as np, os

def create_nonzero_mask(data):
    """Identica a nnU-Net: maschera True dove almeno un canale è nonzero"""
    assert data.ndim in (3, 4)
    if data.ndim == 4:
        mask = np.zeros(data.shape[1:], dtype=bool)
        for c in range(data.shape[0]):
            mask |= data[c] != 0
    else:
        mask = data != 0
    return mask

def crop_to_nonzero(data):
    """Identica a nnU-Net: calcola bounding box e croppa"""
    mask = create_nonzero_mask(data)
    # Bounding box per ogni dimensione
    nonzero = np.where(mask)
    bbox = [(int(np.min(ax)), int(np.max(ax)) + 1) for ax in nonzero]
    slices = tuple(slice(b[0], b[1]) for b in bbox)
    cropped = data[..., slices[0], slices[1]] if data.ndim == 4 else data[slices]
    return cropped, bbox

# Test su immagine reale
img_path = os.path.join(nnUNet_raw, 'Dataset027_ACDC', 'imagesTr',
                         sorted(os.listdir(os.path.join(nnUNet_raw, 'Dataset027_ACDC', 'imagesTr')))[0])
img = nib.load(img_path).get_fdata()
img_ch = img[np.newaxis]  # (1, H, W, D)

n_runs = 10
start = time.time()
for _ in range(n_runs):
    cropped, bbox = crop_to_nonzero(img_ch)
elapsed = (time.time() - start) / n_runs * 1000

print(f'Shape originale:  {img_ch.shape}')
print(f'Shape dopo crop:  {cropped.shape}')
print(f'Tempo medio crop: {elapsed:.1f} ms')
print(f'Metodo bbox:      threshold nonzero su tutti i canali, puro numpy')
print(f'Bounding box:     {bbox}')

Shape originale:  (1, 216, 256, 10)
Shape dopo crop:  (1, 216, 216, 10)
Tempo medio crop: 12.9 ms
Metodo bbox:      threshold nonzero su tutti i canali, puro numpy
Bounding box:     [(0, 216), (0, 256), (0, 10)]


In [26]:
import json, os

trainer_path = '/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py'

# Leggi il file
with open(trainer_path, 'r') as f:
    content = f.read()

# Sostituisci il numero di epoche
content = content.replace('self.num_epochs = 1000', 'self.num_epochs = 150')

# Riscrivi
with open(trainer_path, 'w') as f:
    f.write(content)

print('Epoche modificate a 150.')

Epoche modificate a 150.


In [30]:
!nnUNetv2_train 27 2d 0 --npz --c


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-15 08:43:59.600199: Using torch.compile...
2026-05-15 08:44:10.016114: do_dummy_2d_data_aug: False
2026-05-15 08:44:10.021925: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset027_ACDC/splits_final.json
2026-05-15 08:44:10.026658: The split file contains 5 split

## Results
Run after training is complete.

In [32]:
import json, os

results_fold0 = os.path.join(
    nnUNet_results,
    'Dataset027_ACDC',
    'nnUNetTrainer__nnUNetPlans__2d',
    'fold_0'
)

summary_path = os.path.join(results_fold0, 'validation', 'summary.json')

with open(summary_path, 'r') as f:
    summary = json.load(f)

print('=== Risultati Baseline nnU-Net 2D (fold 0) ===')
per_class = summary.get('mean', {})
label_map = {'1': 'RV', '2': 'MYO', '3': 'LV'}
for label_id, label_name in label_map.items():
    metrics = per_class.get(label_id, {})
    dice = metrics.get('Dice', 'N/A')
    hd95 = metrics.get('HD95', 'N/A')
    dice_str = f'{float(dice):.4f}' if dice != 'N/A' else 'N/A'
    hd95_str = f'{float(hd95):.2f}mm' if hd95 != 'N/A' and str(hd95) != 'nan' else 'N/A'
    print(f'  {label_name}: Dice={dice_str}, HD95={hd95_str}')

# Stampa anche la struttura raw per capire cosa c'è nel json
print('\n--- struttura raw ---')
print(json.dumps(per_class, indent=2))

=== Risultati Baseline nnU-Net 2D (fold 0) ===
  RV: Dice=0.9119, HD95=N/A
  MYO: Dice=0.9012, HD95=N/A
  LV: Dice=0.9405, HD95=N/A

--- struttura raw ---
{
  "1": {
    "Dice": 0.9119122255251166,
    "FN": 364.55,
    "FP": 368.9,
    "IoU": 0.8412207456433478,
    "TN": 464720.85,
    "TP": 4523.3,
    "n_pred": 4892.2,
    "n_ref": 4887.85
  },
  "2": {
    "Dice": 0.9012312972338373,
    "FN": 608.9,
    "FP": 574.875,
    "IoU": 0.8212397763636146,
    "TN": 463329.6,
    "TP": 5464.225,
    "n_pred": 6039.1,
    "n_ref": 6073.125
  },
  "3": {
    "Dice": 0.9404542731270797,
    "FN": 262.2,
    "FP": 233.9,
    "IoU": 0.8936100684920258,
    "TN": 463937.25,
    "TP": 5544.25,
    "n_pred": 5778.15,
    "n_ref": 5806.45
  }
}


In [36]:
import os
import numpy as np
import nibabel as nib

def get_border(mask):
    """Bordo della maschera senza scipy — erosione manuale con shift"""
    eroded = np.zeros_like(mask)
    # Un pixel è interno solo se tutti i vicini diretti sono True
    eroded[1:-1, 1:-1] = (
        mask[1:-1, 1:-1] & mask[:-2, 1:-1] & mask[2:, 1:-1] &
        mask[1:-1, :-2]  & mask[1:-1, 2:]
    )
    return mask & ~eroded

def hd95_numpy(pred, gt, spacing_mm=1.5625):
    if pred.sum() == 0 and gt.sum() == 0:
        return 0.0
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan

    pred_pts = np.argwhere(get_border(pred)).astype(float) * spacing_mm
    gt_pts   = np.argwhere(get_border(gt)).astype(float)   * spacing_mm

    # Distanze punto a punto (chunked per memoria)
    def min_dists(a, b, chunk=500):
        dists = []
        for i in range(0, len(a), chunk):
            diff = a[i:i+chunk, None, :] - b[None, :, :]
            dists.append(np.sqrt((diff**2).sum(-1)).min(axis=1))
        return np.concatenate(dists)

    d1 = min_dists(pred_pts, gt_pts)
    d2 = min_dists(gt_pts, pred_pts)
    return float(np.percentile(np.concatenate([d1, d2]), 95))

# Path
val_dir = os.path.join(
    nnUNet_results, 'Dataset027_ACDC',
    'nnUNetTrainer__nnUNetPlans__2d', 'fold_0', 'validation'
)
labels_dir = os.path.join(nnUNet_raw, 'Dataset027_ACDC', 'labelsTr')

pred_files = sorted([f for f in os.listdir(val_dir) if f.endswith('.nii.gz')])
print(f'Casi in validation: {len(pred_files)}')

hd95_scores = {1: [], 2: [], 3: []}
label_names = {1: 'RV', 2: 'MYO', 3: 'LV'}

for fname in pred_files:
    pred = nib.load(os.path.join(val_dir, fname)).get_fdata()
    gt   = nib.load(os.path.join(labels_dir, fname)).get_fdata()
    for label in [1, 2, 3]:
        # Calcola slice per slice (immagini 3D)
        scores_case = []
        for s in range(pred.shape[2]):
            score = hd95_numpy(pred[:,:,s] == label, gt[:,:,s] == label)
            if not np.isnan(score):
                scores_case.append(score)
        if scores_case:
            hd95_scores[label].append(np.mean(scores_case))

print('\n=== HD95 (mm) ===')
for label, scores in hd95_scores.items():
    print(f'  {label_names[label]}: {np.mean(scores):.2f} mm  ({len(scores)} casi)')

Casi in validation: 40

=== HD95 (mm) ===
  RV: 2.99 mm  (40 casi)
  MYO: 2.79 mm  (40 casi)
  LV: 2.48 mm  (40 casi)


In [34]:
import torch, os

checkpoint_path = os.path.join(results_fold0, 'checkpoint_final.pth')
export_path     = '/content/drive/MyDrive/tesi/baseline_weights.pth'

if not os.path.exists(checkpoint_path):
    print('checkpoint_final.pth non trovato. Completa il training prima.')
else:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model_state = checkpoint['network_weights']
    torch.save({
        'model_state_dict': model_state,
        'epoch': checkpoint.get('current_epoch', 'unknown')
    }, export_path)
    size_mb = os.path.getsize(export_path) / 1e6
    print(f'Pesi salvati: {export_path} ({size_mb:.1f} MB)')
    print('\nPrime 5 chiavi del modello:')
    for k in list(model_state.keys())[:5]:
        print(f'  {k}: {model_state[k].shape}')

Pesi salvati: /content/drive/MyDrive/tesi/baseline_weights.pth (82.6 MB)

Prime 5 chiavi del modello:
  encoder.stages.0.0.convs.0.conv.weight: torch.Size([32, 1, 3, 3])
  encoder.stages.0.0.convs.0.conv.bias: torch.Size([32])
  encoder.stages.0.0.convs.0.norm.weight: torch.Size([32])
  encoder.stages.0.0.convs.0.norm.bias: torch.Size([32])
  encoder.stages.0.0.convs.0.all_modules.0.weight: torch.Size([32, 1, 3, 3])
